# Збір реальних метаданих: реклама в ШІ-системах та її вплив на користувачів

Ноутбук збирає метадані наукових публікацій з **OpenAlex**, **Crossref** та **arXiv**
(відкриті API, ключі не потрібні), зливає дублікати, кодує кожну роботу за схемою
`ai_ads_meta/taxonomy.py` і зберігає результат у файли для подальшого аналізу.

Порядок роботи: виконайте комірки згори донизу. Параметри збору — у розділі 2.
Потрібні `pandas` та доступ до інтернету.

## 1. Підключення пакета `ai_ads_meta`

Якщо ноутбук запущено поза репозиторієм (наприклад, у Colab), пакет буде склоновано з GitHub.

In [1]:
import pathlib
import subprocess
import sys

REPO = "https://github.com/omega2417/bnt.git"
BRANCH = "claude/cool-sagan-b4s8zp"

def find_repo_root():
    here = pathlib.Path.cwd()
    for d in [here, *here.parents]:
        if (d / "ai_ads_meta" / "sources.py").exists():
            return d
    return None

root = find_repo_root()
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "bnt"], check=True)
    root = pathlib.Path("bnt").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd

from ai_ads_meta import analyze, sources
from ai_ads_meta.model import dedupe, load_jsonl, save_jsonl
from ai_ads_meta.taxonomy import TAXONOMY, classify

print("Пакет:", root / "ai_ads_meta")

Пакет: /home/user/bnt/ai_ads_meta


## 2. Параметри збору

- `QUERIES` — пошукові запити (англійською: так індексовано більшість робіт).
- `MAX_PER_QUERY` — скільки записів брати на один запит з кожного джерела.
  50 — швидка проба (~2 хв); для повного корпусу ставте 200–500.
- `MAILTO` — необов'язкова контактна адреса: OpenAlex і Crossref дають швидший
  доступ клієнтам, які її вказали. Адреса передається цим сервісам у заголовку запиту.

In [2]:
import datetime

QUERIES = list(sources.DEFAULT_QUERIES)
# QUERIES += ["algorithmic anomie advertising", "AI companion advertising wellbeing"]

SOURCES = ["openalex", "crossref", "arxiv"]
FROM_YEAR = 2018
MAX_PER_QUERY = 50
MIN_RELEVANCE = 1      # 6 і більше відсікає слабко релевантні записи
MAILTO = ""

OUT_DIR = pathlib.Path("ai_ads_out") / datetime.date.today().isoformat()
OUT_DIR.mkdir(parents=True, exist_ok=True)

for q in QUERIES:
    print("-", q)
print("\nРезультати буде збережено в", OUT_DIR.resolve())

- advertising large language models
- sponsored content conversational AI
- chatbot advertising user trust
- native advertising generative AI
- LLM sponsored product recommendation
- generative AI advertising disclosure consumers
- AI-generated advertising persuasion
- conversational agent advertising manipulation
- personalized advertising AI privacy autonomy
- generative engine optimization brand

Результати буде збережено в /home/user/bnt/ai_ads_meta/ai_ads_out/2026-11-24


## 3. Збір даних

Запит, що завершився помилкою, пропускається (рядок `FAILED`), решта даних зберігається.

In [3]:
raw = sources.collect(
    QUERIES, SOURCES, sources.make_fetch(MAILTO),
    from_year=FROM_YEAR, limit=MAX_PER_QUERY, log=sys.stdout,
)
print(f"\nЗібрано записів: {len(raw)}")
assert raw, "Нічого не зібрано: перевірте доступ до мережі та рядки FAILED вище"

[openalex] 'advertising large language models': 50 records


[openalex] 'sponsored content conversational AI': 50 records


[openalex] 'chatbot advertising user trust': 50 records


[openalex] 'native advertising generative AI': 50 records


[openalex] 'LLM sponsored product recommendation': 50 records


[openalex] 'generative AI advertising disclosure consumers': 50 records


... (20 outputs left out)

[arxiv] 'AI-generated advertising persuasion': 8 records


[arxiv] 'conversational agent advertising manipulation': 1 records


[arxiv] 'personalized advertising AI privacy autonomy': 1 records


[arxiv] 'generative engine optimization brand': 18 records

Зібрано записів: 1090


## 4. Злиття дублікатів, кодування, збереження

In [4]:
records = [classify(r) for r in dedupe(raw)]
save_jsonl(records, OUT_DIR / "records.jsonl")
summary = analyze.run(records, str(OUT_DIR), MIN_RELEVANCE)

print(f"Унікальних записів: {summary['collected']}")
print(f"Релевантних (згадано і ШІ, і рекламу): {summary['relevant']}")
print("\nФайли:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:16} {f.stat().st_size / 1024:8.1f} КБ")

Унікальних записів: 864
Релевантних (згадано і ШІ, і рекламу): 315

Файли:
  features.csv         16.6 КБ
  records.jsonl      1642.8 КБ
  relevant.csv        657.3 КБ
  report.md             7.1 КБ
  summary.json         11.7 КБ


| Файл | Зміст |
|---|---|
| `records.jsonl` | усі унікальні записи з кодами |
| `relevant.csv` | релевантні записи, відсортовані за релевантністю |
| `features.csv` | бінарна матриця «публікація × категорія» |
| `summary.json`, `report.md` | зведена статистика |

## 5. Дані як таблиця `pandas`

`df` — усі записи, `rel` — релевантні. Колонки `ad_format`, `user_effect`, `method`,
`authors`, `countries`, `keywords` містять списки.

In [5]:
df = pd.DataFrame([r.to_dict() for r in records])
rel = df[df["relevant"] & (df["relevance"] >= MIN_RELEVANCE)].sort_values(
    ["relevance", "cited_by"], ascending=False).reset_index(drop=True)

print(df.shape, "->", rel.shape)
rel[["title", "year", "venue", "cited_by", "relevance", "ad_format", "user_effect", "method", "framing"]].head(15)

(864, 23) -> (315, 23)


,title,year,venue,cited_by,relevance,ad_format,user_effect,method,framing
0,Computational Advertising for Meaningful Brand...,2024,Journal of Current Issues & Research in Advert...,9.0,20,[],[],[],benefit
1,Ads that Talk Back: Implications and Perceptio...,2025,Proceedings of the ACM on Interactive Mobile W...,49.0,18,"[conversational_ads, targeting_personalization...",[],"[experiment, system_audit]",benefit
2,Real-time Ad Retrieval via LLM-generative Comm...,2025,Conference on Empirical Methods in Natural Lan...,12.0,18,[],[purchase_behavior],[experiment],benefit
3,Designing brand chatbots: The impact of chatbo...,2019,Aaltodoc (Aalto University),4.0,18,[],[],"[survey, qualitative]",neutral
4,Human vs. machine as message source in adverti...,2022,MOspace Institutional Repository (University o...,2.0,18,[ai_generated_creative],"[trust_credibility, manipulation_persuasion, p...","[experiment, survey]",benefit
5,Proactive Hybrid Recommendation Advertising in...,2026,Zenodo (CERN European Organization for Nuclear...,0.0,18,"[conversational_ads, recommendation_bias, disc...","[privacy, autonomy_agency]","[experiment, conceptual_legal]",mixed
6,Proactive Hybrid In-Stream Advertising in Gene...,2026,Zenodo (CERN European Organization for Nuclear...,0.0,18,"[conversational_ads, disclosure_labeling]","[privacy, autonomy_agency]","[experiment, conceptual_legal]",benefit
7,The Dark Side of Generative AI in Marketing: A...,2026,Zenodo (CERN European Organization for Nuclear...,0.0,18,"[disclosure_labeling, covert_dark_patterns]","[trust_credibility, manipulation_persuasion, p...",[survey],risk
8,Proactive Hybrid Recommendation Advertising: A...,2026,Zenodo (CERN European Organization for Nuclear...,0.0,17,"[conversational_ads, disclosure_labeling]",[],"[experiment, conceptual_legal]",benefit
9,LLM-Auction: Generative Auction towards LLM-Na...,2025,arXiv,24.0,16,[conversational_ads],[satisfaction_engagement],[experiment],benefit


## 6. Швидка перевірка зібраного

Коди присвоюються за ключовими словами, тож одна робота може мати кілька кодів, а частина — жодного.

In [6]:
print("Заповненість полів серед релевантних записів, %")
filled = pd.Series({
    "анотація": (rel["abstract"].str.len() > 0).mean(),
    "DOI": (rel["doi"].str.len() > 0).mean(),
    "країни авторів": (rel["countries"].str.len() > 0).mean(),
    "кількість цитувань": rel["cited_by"].notna().mean(),
    "код впливу на користувачів": (rel["user_effect"].str.len() > 0).mean(),
}).mul(100).round(1)
display(filled.to_frame("%"))

display(rel["year"].value_counts().sort_index().to_frame("публікацій"))

Заповненість полів серед релевантних записів, %


,%
анотація,95.9
DOI,76.8
країни авторів,55.2
кількість цитувань,82.2
код впливу на користувачів,69.8


,публікацій
year,
2018,2
2019,7
2020,1
2021,3
2022,2
2023,11
2024,34
2025,72
2026,183


In [7]:
def code_counts(column):
    counts = rel[column].explode().value_counts()
    return pd.DataFrame({"n": counts, "%": (100 * counts / len(rel)).round(1)})

for column in TAXONOMY:
    display(code_counts(column).rename_axis(column))

,n,%
ad_format,,
disclosure_labeling,116,36.8
ai_generated_creative,77,24.4
recommendation_bias,75,23.8
targeting_personalization,60,19.0
conversational_ads,37,11.7
covert_dark_patterns,13,4.1


,n,%
user_effect,,
trust_credibility,100,31.7
purchase_behavior,75,23.8
manipulation_persuasion,69,21.9
satisfaction_engagement,67,21.3
privacy,51,16.2
autonomy_agency,40,12.7
wellbeing,32,10.2
fairness_vulnerable_groups,20,6.3
misinformation,17,5.4


,n,%
method,,
experiment,107,34.0
conceptual_legal,78,24.8
system_audit,56,17.8
survey,44,14.0
qualitative,25,7.9
review,12,3.8


In [8]:
# Таблиця спряженості «формат реклами × вплив на користувачів»
pairs = rel[["ad_format", "user_effect"]].explode("ad_format").explode("user_effect").dropna().reset_index(drop=True)
pd.crosstab(pairs["ad_format"], pairs["user_effect"])

user_effect,autonomy_agency,fairness_vulnerable_groups,manipulation_persuasion,misinformation,privacy,purchase_behavior,satisfaction_engagement,social_norms_anomie,trust_credibility,wellbeing
ad_format,,,,,,,,,,
ai_generated_creative,7,2,30,3,8,31,18,1,41,15
conversational_ads,7,1,6,3,4,2,7,0,5,1
covert_dark_patterns,1,1,8,3,1,6,1,0,5,0
disclosure_labeling,21,11,31,7,27,44,29,0,64,13
recommendation_bias,9,6,9,7,10,10,13,0,15,3
targeting_personalization,20,10,16,4,31,25,25,0,32,9


## 7. Подальший аналіз

Щоб повернутися до вже зібраних даних без повторного збору:

```python
records = load_jsonl("ai_ads_out/РРРР-ММ-ДД/records.jsonl")
features = pd.read_csv("ai_ads_out/РРРР-ММ-ДД/features.csv")
```

Перед використанням кодів у статті перевірте вручну вибірку записів з `relevant.csv`
і за потреби відкоригуйте правила в `ai_ads_meta/taxonomy.py`.